# Notebook 05 — Late Fusion: Engineered Features + Frozen Prithvi Embeddings (XGBoost)
**Runtime:** Google Colab — **GPU (T4 minimum, A100 preferred)**

**Purpose:** Late-fusion ablation — concatenate hand-engineered contextual proxies (35-d)
with frozen Prithvi-EO-2.0 embeddings (1024-d) and train an XGBoost head on the combined
1059-d representation. This tests whether tabular socioeconomic proxies and geospatial
foundation-model features provide complementary information under geographic distribution shift.

Three feature sets are compared:
1. **Engineered only** (35 features) — baseline
2. **Prithvi embedding only** (1024-d) — from prior notebooks
3. **Engineered + Prithvi fusion** (1059 features) — this notebook's main experiment

- Encoder: `prithvi_eo_v2_300` (frozen) via TerraTorch
- Head: `XGBRegressor(objective="reg:squarederror")` — one per target
- Tuning: Tier 1 hyperparameters + `reg_lambda`, 48 random configurations, 5-fold stratified CV

Two models are trained sequentially — one per aggregate target:

| Target | Formula | Interpretation |
|---|---|---|
| `coverage_fraction` | n_ookla_tiles_in_chip / total_possible_zoom16_tiles | Spatial coverage |
| `log_mean_tests` | log(1 + total_tests / n_subtiles) | Test density per sub-tile |

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/sampled_patches.csv` — patch metadata + binary labels
- `data/raw/ookla_india_2019_Q1.csv` — raw Ookla sub-tile data (from Google Drive if needed)

## Step 0: Environment Setup

> **Note:** The cell below installs `terratorch` and pins NumPy.  
> After it runs the runtime will **restart automatically**.  
> Re-run from **Step 0.2** after the restart.

In [ ]:
# Cell 0.1: Install dependencies
# TerraTorch MUST be installed first — restart runtime after this cell.

!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q xgboost rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow joblib

import os
print("Restarting runtime to apply numpy pin — re-run from Step 0.2 after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# Cell 0.2: Clone repo

import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# Cell 0.3: Sync patches from Google Drive

from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')


# Verify patches are available
patch_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if patch_count >= 6900:
    print(f'Patches available: {patch_count:,}')
else:
    raise FileNotFoundError(
        f'Only {patch_count} patches found in {PATCH_DIR}. '
        f'Run NB01 first (Steps 1-7) to download patches and save them to Google Drive.'
    )

## Step 1: Imports, Constants & Reproducibility

In [ ]:
import os
import random
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import torch
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging

from shapely.geometry import box
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterSampler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, root_mean_squared_error,
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix,
    classification_report,
)
from scipy.stats import spearmanr, binned_statistic
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

# ── Deterministic seeding ────────────────────────────────────
SEED = 42

def seed_everything(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

seed_everything(SEED)

# ── Constants (shared across notebooks) ──────────────────────
HLS_MEANS = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS  = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE     = 0.0001
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']

PATCH_DIR = 'data/raw/patches_2019'

OUTPUT_FEATURES = Path("outputs/features")
OUTPUT_RESULTS  = Path("outputs/results")
OUTPUT_MODELS   = Path("outputs/models")
OUTPUT_FIGURES  = Path("outputs/figures")

for p in [OUTPUT_FEATURES, OUTPUT_RESULTS, OUTPUT_MODELS, OUTPUT_FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

PATCH_SIZE_M   = 6720.0
PATCH_SIZE_DEG = PATCH_SIZE_M / 111_320.0
EARTH_CIRC_M   = 40_075_016.686
ZOOM16_SIDE_EQ = EARTH_CIRC_M / (2 ** 16)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels,
aggregate targets (`coverage_fraction`, `log_mean_tests`),
stratification features, and geographic split.

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

## Step 5: Engineered Feature Extraction (same 35 engineered features)

In [ ]:
def extract_engineered_features(patch_path):
    with rasterio.open(patch_path) as src:
        img = src.read().astype(np.float32) * SCALE
    feats = {}
    for i, name in enumerate(BAND_NAMES):
        band  = img[i]
        valid = band[band > 0]
        if len(valid) == 0: valid = np.array([0.0])
        feats[f'{name}_mean'] = float(valid.mean())
        feats[f'{name}_std']  = float(valid.std())
        feats[f'{name}_p10']  = float(np.percentile(valid, 10))
        feats[f'{name}_p90']  = float(np.percentile(valid, 90))
    red, nir, swir1, green = img[2], img[3], img[4], img[1]
    ndvi  = np.where((nir+red)    > 0, (nir-red)    / (nir+red    +1e-8), 0)
    ndbi  = np.where((swir1+nir)  > 0, (swir1-nir)  / (swir1+nir  +1e-8), 0)
    mndwi = np.where((green+swir1)> 0, (green-swir1)/ (green+swir1+1e-8), 0)
    feats['NDVI_mean']       = float(ndvi.mean())
    feats['NDVI_std']        = float(ndvi.std())
    feats['NDBI_mean']       = float(ndbi.mean())
    feats['NDBI_std']        = float(ndbi.std())
    feats['MNDWI_mean']      = float(mndwi.mean())
    feats['NIR_spatial_var'] = float(nir.var())
    feats['bright_frac']     = float((red > 0.15).mean())
    return feats

def build_feature_matrix(df, patch_dir, desc=''):
    rows, failed = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:    rows.append(extract_engineered_features(f"{patch_dir}/{row['patch_id']}.tif"))
        except: failed.append(row['patch_id']); rows.append({})
    if failed: print(f'  Warning: {len(failed)} failed')
    return pd.DataFrame(rows)

print('Extracting engineered features ...')
X_train_eng = build_feature_matrix(train_df, PATCH_DIR, 'Train')
X_val_eng   = build_feature_matrix(val_df,   PATCH_DIR, 'Val')
X_test_eng  = build_feature_matrix(test_df,  PATCH_DIR, 'Test')

# Add contextual features (ntl, builtup, elevation) from metadata
for feat in ['ntl', 'builtup', 'elevation']:
    X_train_eng[feat] = train_df[feat].values
    X_val_eng[feat]   = val_df[feat].values
    X_test_eng[feat]  = test_df[feat].values

# Fill NaN with train medians
col_med = X_train_eng.median()
X_train_eng = X_train_eng.fillna(col_med)
X_val_eng   = X_val_eng.fillna(col_med)
X_test_eng  = X_test_eng.fillna(col_med)

ENGINEERED_COLS = list(X_train_eng.columns)
print(f'Engineered feature matrix: {X_train_eng.shape}  ({len(ENGINEERED_COLS)} features)')
print(f'Features: {ENGINEERED_COLS}')

## Step 6: Frozen Prithvi Embedding Extraction

Extracts 1024-d mean-pooled embeddings from the frozen `prithvi_eo_v2_300` encoder.
If prior notebooks have already cached embeddings, they can be loaded instead; otherwise we extract fresh.

In [ ]:
class PrithviPatchDataset(Dataset):
    def __init__(self, df, patch_dir, n_bands=6):
        self.df        = df.reset_index(drop=True)
        self.patch_dir = patch_dir
        self.n_bands   = n_bands

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f"{self.patch_dir}/{row['patch_id']}.tif"
        with rasterio.open(path) as src:
            img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
        img *= SCALE
        for b in range(self.n_bands):
            img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
        img = np.nan_to_num(img, nan=0.0)
        return torch.from_numpy(img).unsqueeze(1), row['patch_id']

In [ ]:
# ── Try loading cached embeddings; check local then Drive ────
DRIVE_FEATURES = Path('/content/drive/MyDrive/prithvi_embeddings')
DRIVE_FEATURES.mkdir(parents=True, exist_ok=True)


def load_cached_embeddings(split_name):
    local_path = OUTPUT_FEATURES / f"prithvi_frozen_embeds_{split_name}.npz"
    drive_path = DRIVE_FEATURES / f"prithvi_frozen_embeds_{split_name}.npz"

    if local_path.exists():
        data = np.load(local_path)
        print(f'  Loaded cached {split_name} embeddings (local): {data["X"].shape}')
        return data["X"]

    if drive_path.exists():
        import shutil
        shutil.copy(drive_path, local_path)
        data = np.load(local_path)
        print(f'  Loaded cached {split_name} embeddings (Drive): {data["X"].shape}')
        return data["X"]

    return None

X_train_emb = load_cached_embeddings("train")
X_val_emb   = load_cached_embeddings("val")
X_test_emb  = load_cached_embeddings("test")

NEED_EXTRACTION = any(x is None for x in [X_train_emb, X_val_emb, X_test_emb])

if NEED_EXTRACTION:
    print('\nCached embeddings not found — extracting from scratch ...')
else:
    print('\nAll embeddings loaded from cache.')

In [ ]:
# ── Extract embeddings if not cached ─────────────────────────
if NEED_EXTRACTION:
    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False
    print(f'Prithvi encoder loaded on {DEVICE}')

    @torch.no_grad()
    def extract_prithvi_embeddings(df, batch_size=64):
        ds = PrithviPatchDataset(df, PATCH_DIR)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
        all_embs = []
        for x, _ in tqdm(loader, desc='Extracting embeddings'):
            x = x.to(DEVICE, dtype=torch.float32, non_blocking=True)
            feats = encoder(x)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))
        return np.concatenate(all_embs, axis=0)

    X_train_emb = extract_prithvi_embeddings(train_df)
    X_val_emb   = extract_prithvi_embeddings(val_df)
    X_test_emb  = extract_prithvi_embeddings(test_df)

    # Cache for future use
    for split_name, X_emb, df in [("train", X_train_emb, train_df),
                                    ("val", X_val_emb, val_df),
                                    ("test", X_test_emb, test_df)]:
        np.savez_compressed(
            OUTPUT_FEATURES / f"prithvi_frozen_embeds_{split_name}.npz",
            X=X_emb,
            patch_id=df["patch_id"].to_numpy(),
            lon=df["lon"].to_numpy(),
            lat=df["lat"].to_numpy(),
            has_coverage=df["has_coverage"].to_numpy(),
            coverage_fraction=df["coverage_fraction"].to_numpy(),
            log_mean_tests=df["log_mean_tests"].to_numpy(),
        )
    print('Embeddings extracted and cached.')

    del encoder
    torch.cuda.empty_cache()
    print('Encoder released from GPU.')
else:
    print('Using cached embeddings — no GPU extraction needed.')

print(f'\nPrithvi embedding shape: train={X_train_emb.shape}, val={X_val_emb.shape}, test={X_test_emb.shape}')


    # Sync extracted embeddings to Drive for persistence
    import shutil
    for split_name in ['train', 'val', 'test']:
        local_p = OUTPUT_FEATURES / f'prithvi_frozen_embeds_{split_name}.npz'
        drive_p = DRIVE_FEATURES / f'prithvi_frozen_embeds_{split_name}.npz'
        if local_p.exists() and not drive_p.exists():
            shutil.copy(local_p, drive_p)
    print('Embeddings synced to Google Drive.')

## Step 7: Build Fusion Features

Concatenate engineered features (35-d) with frozen Prithvi embeddings (1024-d) to form
the 1059-d late-fusion representation.

In [ ]:
# ── Engineered-only arrays (for comparison) ──────────────────
X_train_eng_np = X_train_eng.values.astype(np.float32)
X_val_eng_np   = X_val_eng.values.astype(np.float32)
X_test_eng_np  = X_test_eng.values.astype(np.float32)

# ── Prithvi-only arrays ─────────────────────────────────────
X_train_prithvi = X_train_emb.astype(np.float32)
X_val_prithvi   = X_val_emb.astype(np.float32)
X_test_prithvi  = X_test_emb.astype(np.float32)

# ── Fusion arrays (engineered + Prithvi) ────────────────────
X_train_fusion = np.hstack([X_train_eng_np, X_train_prithvi])
X_val_fusion   = np.hstack([X_val_eng_np,   X_val_prithvi])
X_test_fusion  = np.hstack([X_test_eng_np,  X_test_prithvi])

# Feature names for the fusion representation
PRITHVI_COLS = [f'prithvi_{i}' for i in range(X_train_prithvi.shape[1])]
FUSION_COLS  = ENGINEERED_COLS + PRITHVI_COLS

print(f'Engineered only : {X_train_eng_np.shape[1]} features')
print(f'Prithvi only    : {X_train_prithvi.shape[1]} features')
print(f'Fusion          : {X_train_fusion.shape[1]} features')
print(f'  = {len(ENGINEERED_COLS)} engineered + {len(PRITHVI_COLS)} Prithvi')

## Step 8: Hyperparameter Search & Training Infrastructure

48 random configurations, 5-fold stratified CV.

In [ ]:
# ── Tier 1 + reg_lambda search space ────────────────────────
PARAM_DISTS = {
    "learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "max_depth":        [2, 3, 4, 6],
    "n_estimators":     [300, 600, 1200],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.2, 0.4, 0.8, 1.0],
    "reg_lambda":       [1.0, 3.0, 10.0, 30.0],
}

N_CV_ITER = 48


def make_xgb_regressor(params):
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        device="cpu",
        eval_metric="rmse",
        early_stopping_rounds=50,
        random_state=SEED,
        seed=SEED,
        seed_per_iteration=True,
        n_jobs=8,
        importance_type="gain",
        **params,
    )


def cv_search(X, y_cont, y_bin, n_iter=N_CV_ITER):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    sampled_params = list(ParameterSampler(PARAM_DISTS, n_iter=n_iter, random_state=SEED))
    rows = []

    for i, params in enumerate(tqdm(sampled_params, desc='CV search')):
        fold_rows = []
        for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y_bin), start=1):
            model = make_xgb_regressor(params)
            model.fit(
                X[tr_idx], y_cont[tr_idx],
                eval_set=[(X[va_idx], y_cont[va_idx])],
                verbose=False,
            )
            va_pred = model.predict(X[va_idx])
            fold_rows.append({
                "fold": fold,
                "rmse": root_mean_squared_error(y_cont[va_idx], va_pred),
                "mae": mean_absolute_error(y_cont[va_idx], va_pred),
                "spearman_rho": spearmanr(y_cont[va_idx], va_pred).statistic,
                "pr_auc_binary": average_precision_score(y_bin[va_idx], va_pred),
                "roc_auc_binary": roc_auc_score(y_bin[va_idx], va_pred),
                "best_iteration": getattr(model, "best_iteration", None),
            })

        fr = pd.DataFrame(fold_rows)
        rows.append({
            **params,
            "cv_rmse_mean": fr["rmse"].mean(),
            "cv_rmse_std": fr["rmse"].std(),
            "cv_mae_mean": fr["mae"].mean(),
            "cv_spearman_mean": fr["spearman_rho"].mean(),
            "cv_pr_auc_mean": fr["pr_auc_binary"].mean(),
            "cv_roc_auc_mean": fr["roc_auc_binary"].mean(),
            "cv_best_iteration_mean": fr["best_iteration"].mean(),
        })

    results = pd.DataFrame(rows).sort_values(
        by=["cv_rmse_mean", "cv_spearman_mean"],
        ascending=[True, False]
    ).reset_index(drop=True)
    return results


def choose_threshold_from_validation(y_val_bin, val_scores):
    precision, recall, thresholds = precision_recall_curve(y_val_bin, val_scores)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1))
    return float(thresholds[best_idx])


print(f'Search space: {N_CV_ITER} random configs x 5 folds = {N_CV_ITER * 5} fits per target per feature set')
print(f'Parameters: {list(PARAM_DISTS.keys())}')

## Step 9: Train & Evaluate — All Feature Sets x Both Targets

Runs CV search, refit, threshold selection, and test evaluation for each combination:
- **Engineered only** (35-d)
- **Prithvi only** (1024-d)
- **Fusion** (1059-d)

× 2 targets (`coverage_fraction`, `log_mean_tests`)

In [ ]:
def fit_and_evaluate(feat_label, X_tr, X_va, X_te, target_name,
                     meta_train, meta_val, meta_test, feature_names=None):
    """Full pipeline: CV search → refit → threshold → test metrics."""
    print(f'\n{"="*60}')
    print(f'Feature set: {feat_label}  |  Target: {target_name}')
    print(f'  X_train shape: {X_tr.shape}')
    print(f'{"="*60}')

    y_train = meta_train[target_name].to_numpy(np.float32)
    y_val   = meta_val[target_name].to_numpy(np.float32)
    y_test  = meta_test[target_name].to_numpy(np.float32)

    y_train_bin = meta_train["has_coverage"].to_numpy(np.int32)
    y_val_bin   = meta_val["has_coverage"].to_numpy(np.int32)
    y_test_bin  = meta_test["has_coverage"].to_numpy(np.int32)

    # ── CV search ────────────────────────────────────────────
    print('Running CV search ...')
    search_df = cv_search(X_tr, y_train, y_train_bin, n_iter=N_CV_ITER)

    print('\nTop 5 configurations:')
    display_cols = list(PARAM_DISTS.keys()) + [
        'cv_rmse_mean', 'cv_rmse_std', 'cv_spearman_mean', 'cv_pr_auc_mean'
    ]
    print(search_df[display_cols].head(5).to_string(index=False))

    best_params = search_df.iloc[0][list(PARAM_DISTS.keys())].to_dict()
    best_params['max_depth'] = int(best_params['max_depth'])
    best_params['n_estimators'] = int(best_params['n_estimators'])
    print(f'\nBest params: {best_params}')

    # ── Refit ────────────────────────────────────────────────
    model = make_xgb_regressor(best_params)
    model.fit(X_tr, y_train,
              eval_set=[(X_tr, y_train), (X_va, y_val)],
              verbose=False)
    print(f'Best iteration: {model.best_iteration}')

    # ── Threshold selection ──────────────────────────────────
    val_scores = model.predict(X_va)
    thr_star = choose_threshold_from_validation(y_val_bin, val_scores)
    print(f'Val-optimal threshold: {thr_star:.4f}')

    # ── Test evaluation ──────────────────────────────────────
    test_scores = model.predict(X_te)
    test_pred_bin = (test_scores >= thr_star).astype(np.int32)
    residuals = test_scores - y_test

    mae  = mean_absolute_error(y_test, test_scores)
    rmse = root_mean_squared_error(y_test, test_scores)
    rho  = spearmanr(y_test, test_scores).statistic

    pr_auc   = average_precision_score(y_test_bin, test_scores)
    roc_auc  = roc_auc_score(y_test_bin, test_scores)
    f1_opt   = f1_score(y_test_bin, test_pred_bin)
    prec_opt = precision_score(y_test_bin, test_pred_bin, zero_division=0)
    rec_opt  = recall_score(y_test_bin, test_pred_bin)
    acc_opt  = accuracy_score(y_test_bin, test_pred_bin)

    print(f'\n--- PRIMARY: Regression quality ---')
    print(f'  MAE          : {mae:.4f}')
    print(f'  RMSE         : {rmse:.4f}')
    print(f'  Spearman rho : {rho:.4f}')
    print(f'\n--- SECONDARY: Binary connectivity task ---')
    print(f'  PR-AUC        : {pr_auc:.4f}')
    print(f'  ROC-AUC       : {roc_auc:.4f}')
    print(f'  F1 @ thr={thr_star:.3f} : {f1_opt:.4f}')
    print(f'  Precision     : {prec_opt:.4f}  |  Recall: {rec_opt:.4f}  |  Acc: {acc_opt:.4f}')
    print()
    print(classification_report(y_test_bin, test_pred_bin,
                                 target_names=['No Coverage', 'Has Coverage']))

    metrics = {
        "feature_set": feat_label,
        "target": target_name,
        "n_features": X_tr.shape[1],
        "mae": mae,
        "rmse": rmse,
        "spearman_rho": rho,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "f1_opt": f1_opt,
        "opt_threshold": thr_star,
        "precision_opt": prec_opt,
        "recall_opt": rec_opt,
        "accuracy_opt": acc_opt,
        "best_iteration": getattr(model, "best_iteration", None),
        "n_train": len(meta_train),
        "n_val": len(meta_val),
        "n_test": len(meta_test),
        **{f"best_{k}": v for k, v in best_params.items()},
    }

    # ── Save artifacts ───────────────────────────────────────
    safe_label = feat_label.lower().replace(' ', '_').replace('+', '_')
    search_df.to_csv(OUTPUT_RESULTS / f"fusion_{safe_label}_{target_name}_cv.csv", index=False)
    pd.DataFrame([metrics]).to_csv(OUTPUT_RESULTS / f"fusion_{safe_label}_{target_name}_metrics.csv", index=False)
    model.save_model(str(OUTPUT_MODELS / f"fusion_{safe_label}_{target_name}.json"))

    return {
        'model': model,
        'search_df': search_df,
        'metrics': metrics,
        'test_scores': test_scores,
        'y_test_cont': y_test,
        'y_test_bin': y_test_bin,
        'residuals': residuals,
        'feature_names': feature_names,
    }

In [ ]:
# ── Define all experiments ───────────────────────────────────
FEATURE_SETS = {
    "Engineered only": {
        "X_train": X_train_eng_np, "X_val": X_val_eng_np, "X_test": X_test_eng_np,
        "feature_names": ENGINEERED_COLS,
    },
    "Prithvi only": {
        "X_train": X_train_prithvi, "X_val": X_val_prithvi, "X_test": X_test_prithvi,
        "feature_names": PRITHVI_COLS,
    },
    "Engineered+Prithvi": {
        "X_train": X_train_fusion, "X_val": X_val_fusion, "X_test": X_test_fusion,
        "feature_names": FUSION_COLS,
    },
}

TARGETS = ['coverage_fraction', 'log_mean_tests']

# ── Run all experiments ──────────────────────────────────────
all_results = {}

for feat_label, feat_data in FEATURE_SETS.items():
    for target in TARGETS:
        key = (feat_label, target)
        all_results[key] = fit_and_evaluate(
            feat_label=feat_label,
            X_tr=feat_data["X_train"],
            X_va=feat_data["X_val"],
            X_te=feat_data["X_test"],
            target_name=target,
            meta_train=train_df,
            meta_val=val_df,
            meta_test=test_df,
            feature_names=feat_data["feature_names"],
        )

print('\n\nAll experiments complete.')

## Step 10: Diagnostic Plots — All Feature Sets

Full diagnostic suite for **each feature set** (Engineered only, Prithvi only, Fusion)
on each target. Figures 1–5 are produced per feature set × target; Figure 6 (feature
importance by source) is only shown for the fusion model.

In [ ]:
PLOT_SETS = {
    "Engineered only":    "engineered_only",
    "Prithvi only":       "prithvi_only",
    "Engineered+Prithvi": "eng_prithvi",
}

for feat_label, safe_label in PLOT_SETS.items():
    for TARGET_NAME in TARGETS:
        r = all_results[(feat_label, TARGET_NAME)]
        model       = r['model']
        metrics     = r['metrics']
        test_scores = r['test_scores']
        y_test_cont = r['y_test_cont']
        y_test_bin  = r['y_test_bin']
        residuals   = r['residuals']
        opt_thr     = metrics['opt_threshold']

        test_pred_bin = (test_scores >= opt_thr).astype(int)
        mae  = metrics['mae']
        rmse = metrics['rmse']
        rho  = metrics['spearman_rho']
        pr_auc   = metrics['pr_auc']
        prec_opt = metrics['precision_opt']
        rec_opt  = metrics['recall_opt']

        tag = f'{safe_label}_{TARGET_NAME}'
        title_prefix = f'{feat_label}'

        print(f'\n{"="*60}')
        print(f'Plots for: {feat_label} — {TARGET_NAME}')
        print(f'{"="*60}')

        # ── Figure 1: Training curve ─────────────────────────
        evals = model.evals_result()
        train_rmse = evals['validation_0']['rmse']
        val_rmse   = evals['validation_1']['rmse']
        best_round = model.best_iteration

        fig, ax = plt.subplots(figsize=(8, 4))
        rounds = range(1, len(train_rmse) + 1)
        ax.plot(rounds, train_rmse, lw=1.5, color='#55A868', alpha=0.7, label='Train RMSE')
        ax.plot(rounds, val_rmse,   lw=2,   color='#4C72B0', label='Val RMSE')
        ax.axvline(best_round + 1, linestyle='--', color='red', alpha=0.8,
                   label=f'Best round {best_round + 1}  (Val RMSE = {val_rmse[best_round]:.4f})')
        ax.set_xlabel('Boosting Round'); ax.set_ylabel('RMSE')
        ax.set_title(f'Training Curve — {title_prefix} — {TARGET_NAME}')
        ax.legend(); plt.tight_layout()
        plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_training_curve.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Figure 2: Predicted vs actual + Residual histogram
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(f'Regression Quality — {title_prefix} — {TARGET_NAME}',
                     fontsize=13, fontweight='bold')

        axes[0].scatter(y_test_cont, test_scores, alpha=0.35, s=12, color='#4C72B0')
        mn = min(y_test_cont.min(), test_scores.min())
        mx = max(y_test_cont.max(), test_scores.max())
        axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='45° reference')
        axes[0].set_xlabel(f'Actual {TARGET_NAME}')
        axes[0].set_ylabel(f'Predicted {TARGET_NAME}')
        axes[0].set_title(
            f'Predicted vs Actual\nMAE={mae:.3f}  RMSE={rmse:.3f}  Spearman ρ={rho:.3f}')
        axes[0].legend(fontsize=9)

        axes[1].hist(residuals, bins=40, color='#4C72B0', edgecolor='white', alpha=0.8)
        axes[1].axvline(0, color='red', linestyle='--', lw=1.5, label='Zero')
        axes[1].axvline(residuals.mean(), color='orange', linestyle='-', lw=1.5,
                        label=f'Mean = {residuals.mean():.3f}')
        axes[1].set_xlabel('Residual (predicted − true)')
        axes[1].set_ylabel('Count')
        axes[1].set_title(f'Residual Distribution  (std = {residuals.std():.3f})')
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_regression.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Figure 3: Residuals vs true value ────────────────
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(y_test_cont, residuals, alpha=0.35, s=12, color='#4C72B0')
        ax.axhline(0, color='red', linestyle='--', lw=1.5, label='Zero residual')
        try:
            bin_means, bin_edges, _ = binned_statistic(
                y_test_cont, residuals, statistic='mean', bins=10)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            ax.plot(bin_centers, bin_means, 'o-', color='orange', lw=2,
                    ms=6, label='Bin mean residual')
        except Exception:
            pass
        ax.set_xlabel(f'True {TARGET_NAME}')
        ax.set_ylabel('Residual  (predicted − true)')
        ax.set_title(f'Residuals vs True Value — {title_prefix} — {TARGET_NAME}')
        ax.legend(); plt.tight_layout()
        plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_residuals.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Figure 4: Spatial prediction map ─────────────────
        lons = test_df['lon'].values
        lats = test_df['lat'].values
        res_abs_max = np.percentile(np.abs(residuals), 95)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(f'Spatial Prediction Map — {title_prefix} — {TARGET_NAME}\nNE India test set',
                     fontsize=13, fontweight='bold')

        sc0 = axes[0].scatter(lons, lats, c=test_scores, cmap='RdYlGn', s=18, alpha=0.75)
        plt.colorbar(sc0, ax=axes[0], label=f'Predicted {TARGET_NAME}')
        axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
        axes[0].set_title('Predicted value')

        sc1 = axes[1].scatter(lons, lats, c=residuals, cmap='RdBu_r', s=18, alpha=0.75,
                              vmin=-res_abs_max, vmax=res_abs_max)
        plt.colorbar(sc1, ax=axes[1], label='Residual (predicted − true)')
        axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
        axes[1].set_title('Residuals  (blue = under-predicted, red = over-predicted)')

        plt.tight_layout()
        plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_spatial.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Figure 5: Confusion matrix + PR curve ────────────
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f'Binary Connectivity Task — {title_prefix} — {TARGET_NAME}',
                     fontsize=13, fontweight='bold')

        cm = confusion_matrix(y_test_bin, test_pred_bin)
        im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
        fig.colorbar(im, ax=axes[0])
        axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
        axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
        for row in range(2):
            for col in range(2):
                axes[0].text(col, row, str(cm[row, col]), ha='center', va='center',
                             color='white' if cm[row, col] > cm.max() / 2 else 'black', fontsize=14)
        axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
        axes[0].set_title(f'Confusion Matrix  (thr = {opt_thr:.3f})')

        prec_t, rec_t, _ = precision_recall_curve(y_test_bin, test_scores)
        axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc:.3f}')
        axes[1].axhline(y_test_bin.mean(), linestyle='--', color='grey',
                        label=f'Baseline ({y_test_bin.mean():.3f})')
        axes[1].scatter([rec_opt], [prec_opt], marker='*', s=200, color='red', zorder=5,
                        label=f'Val-opt thr = {opt_thr:.3f}')
        axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
        axes[1].set_title('Precision-Recall Curve')
        axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

        plt.tight_layout()
        plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_binary.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Figure 6: Feature importance — top 30 ────────────
        importances = model.feature_importances_
        feat_names = r['feature_names'] if r['feature_names'] else [f'f_{i}' for i in range(len(importances))]
        feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

        if feat_label == "Engineered+Prithvi":
            fig, axes = plt.subplots(1, 2, figsize=(14, 7))
            fig.suptitle(f'Feature Importance (gain) — {title_prefix} — {TARGET_NAME}',
                         fontsize=13, fontweight='bold')

            feat_imp.head(30).plot(kind='barh', ax=axes[0], color='steelblue')
            axes[0].set_xlabel('Importance (gain)')
            axes[0].set_title('Top 30 Features')
            axes[0].invert_yaxis()

            eng_imp = feat_imp[[c for c in feat_imp.index if not c.startswith('prithvi_')]].sum()
            pri_imp = feat_imp[[c for c in feat_imp.index if c.startswith('prithvi_')]].sum()
            total = eng_imp + pri_imp
            axes[1].bar(['Engineered\n(35 features)', 'Prithvi\n(1024 dims)'],
                        [eng_imp / total * 100, pri_imp / total * 100],
                        color=['#DD8452', '#4C72B0'])
            axes[1].set_ylabel('Share of total importance (%)')
            axes[1].set_title('Importance by Feature Source')

            plt.tight_layout()
            plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_feature_importance.png',
                        dpi=150, bbox_inches='tight')
            plt.show()
        else:
            fig, ax = plt.subplots(figsize=(10, 7))
            feat_imp.head(30).plot(kind='barh', ax=ax, color='steelblue')
            ax.set_xlabel('Importance (gain)')
            ax.set_title(f'Top 30 Features — {title_prefix} — {TARGET_NAME}')
            ax.invert_yaxis()
            plt.tight_layout()
            plt.savefig(OUTPUT_FIGURES / f'fusion_{tag}_feature_importance.png',
                        dpi=150, bbox_inches='tight')
            plt.show()

print('\nAll diagnostic plots saved.')

## Step 11: Comparison — Engineered vs Prithvi vs Fusion

In [ ]:
# ── Build comparison table from all experiments ──────────────
comp_rows = []
for key, r in all_results.items():
    m = r['metrics']
    comp_rows.append({
        'Feature Set': m['feature_set'],
        'Target': m['target'],
        'Dims': m['n_features'],
        'MAE': m['mae'],
        'RMSE': m['rmse'],
        'Spearman ρ': m['spearman_rho'],
        'PR-AUC': m['pr_auc'],
        'ROC-AUC': m['roc_auc'],
        'F1': m['f1_opt'],
        'Precision': m['precision_opt'],
        'Recall': m['recall_opt'],
        'Accuracy': m['accuracy_opt'],
    })

comparison = pd.DataFrame(comp_rows)
comparison = comparison.sort_values(['Target', 'Feature Set']).reset_index(drop=True)

print('=== Late Fusion Ablation — NE India Test Set ===\n')
print(comparison.to_string(index=False))

comparison.to_csv(OUTPUT_RESULTS / 'fusion_ablation_comparison.csv', index=False)
print('\nComparison table saved.')

In [ ]:
# ── Comparison bar charts per target ─────────────────────────
for target in TARGETS:
    subset = comparison[comparison['Target'] == target].reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Late Fusion Ablation — {target}\nNE India test set',
                 fontsize=13, fontweight='bold')

    labels = subset['Feature Set'].tolist()
    x = np.arange(len(labels))
    w = 0.2

    # Regression metrics
    axes[0].bar(x - w, subset['MAE'],  w, label='MAE',  color='#4C72B0')
    axes[0].bar(x,     subset['RMSE'], w, label='RMSE', color='#DD8452')
    axes[0].bar(x + w, subset['Spearman ρ'].clip(0), w, label='Spearman ρ', color='#55A868')
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=10, ha='right')
    axes[0].set_ylabel('Score'); axes[0].set_title('Regression Metrics')
    axes[0].legend()

    # Binary metrics
    axes[1].bar(x - w, subset['PR-AUC'],  w, label='PR-AUC',  color='#4C72B0')
    axes[1].bar(x,     subset['ROC-AUC'], w, label='ROC-AUC', color='#DD8452')
    axes[1].bar(x + w, subset['F1'],      w, label='F1',      color='#55A868')
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=10, ha='right')
    axes[1].set_ylim(0, 1); axes[1].set_ylabel('Score')
    axes[1].set_title('Binary Connectivity Metrics')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / f'fusion_ablation_{target}_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()

# ── Combined overview figure ─────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 5))
labels = comparison['Feature Set'] + ' — ' + comparison['Target']
x = np.arange(len(labels))
w = 0.25
ax.bar(x - w, comparison['PR-AUC'],  w, label='PR-AUC',  color='#4C72B0')
ax.bar(x,     comparison['ROC-AUC'], w, label='ROC-AUC', color='#DD8452')
ax.bar(x + w, comparison['F1'],      w, label='F1',      color='#55A868')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=18, ha='right')
ax.set_ylim(0, 1); ax.set_ylabel('Score')
ax.set_title('Late Fusion Ablation: Engineered vs Prithvi vs Fusion — All Targets\nNE India test set')
ax.legend(); plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'fusion_ablation_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12: Save Metrics & Push to GitHub

In [ ]:
print('All artifacts:')
for p in sorted(OUTPUT_RESULTS.glob('fusion_*')):
    print(f'  {p}')
for p in sorted(OUTPUT_MODELS.glob('fusion_*')):
    print(f'  {p}')
for p in sorted(OUTPUT_FIGURES.glob('fusion_*')):
    print(f'  {p}')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

files = [
    # Results — all 6 experiments (3 feature sets x 2 targets) + comparison
    'outputs/results/fusion_engineered_only_coverage_fraction_cv.csv',
    'outputs/results/fusion_engineered_only_coverage_fraction_metrics.csv',
    'outputs/results/fusion_engineered_only_log_mean_tests_cv.csv',
    'outputs/results/fusion_engineered_only_log_mean_tests_metrics.csv',
    'outputs/results/fusion_prithvi_only_coverage_fraction_cv.csv',
    'outputs/results/fusion_prithvi_only_coverage_fraction_metrics.csv',
    'outputs/results/fusion_prithvi_only_log_mean_tests_cv.csv',
    'outputs/results/fusion_prithvi_only_log_mean_tests_metrics.csv',
    'outputs/results/fusion_engineered_prithvi_coverage_fraction_cv.csv',
    'outputs/results/fusion_engineered_prithvi_coverage_fraction_metrics.csv',
    'outputs/results/fusion_engineered_prithvi_log_mean_tests_cv.csv',
    'outputs/results/fusion_engineered_prithvi_log_mean_tests_metrics.csv',
    'outputs/results/fusion_ablation_comparison.csv',
    # Models
    'outputs/models/fusion_engineered_only_coverage_fraction.json',
    'outputs/models/fusion_engineered_only_log_mean_tests.json',
    'outputs/models/fusion_prithvi_only_coverage_fraction.json',
    'outputs/models/fusion_prithvi_only_log_mean_tests.json',
    'outputs/models/fusion_engineered_prithvi_coverage_fraction.json',
    'outputs/models/fusion_engineered_prithvi_log_mean_tests.json',
    # Figures — Engineered only diagnostic plots
    'outputs/figures/fusion_engineered_only_coverage_fraction_training_curve.png',
    'outputs/figures/fusion_engineered_only_coverage_fraction_regression.png',
    'outputs/figures/fusion_engineered_only_coverage_fraction_residuals.png',
    'outputs/figures/fusion_engineered_only_coverage_fraction_spatial.png',
    'outputs/figures/fusion_engineered_only_coverage_fraction_binary.png',
    'outputs/figures/fusion_engineered_only_coverage_fraction_feature_importance.png',
    'outputs/figures/fusion_engineered_only_log_mean_tests_training_curve.png',
    'outputs/figures/fusion_engineered_only_log_mean_tests_regression.png',
    'outputs/figures/fusion_engineered_only_log_mean_tests_residuals.png',
    'outputs/figures/fusion_engineered_only_log_mean_tests_spatial.png',
    'outputs/figures/fusion_engineered_only_log_mean_tests_binary.png',
    'outputs/figures/fusion_engineered_only_log_mean_tests_feature_importance.png',
    # Figures — Prithvi only diagnostic plots
    'outputs/figures/fusion_prithvi_only_coverage_fraction_training_curve.png',
    'outputs/figures/fusion_prithvi_only_coverage_fraction_regression.png',
    'outputs/figures/fusion_prithvi_only_coverage_fraction_residuals.png',
    'outputs/figures/fusion_prithvi_only_coverage_fraction_spatial.png',
    'outputs/figures/fusion_prithvi_only_coverage_fraction_binary.png',
    'outputs/figures/fusion_prithvi_only_coverage_fraction_feature_importance.png',
    'outputs/figures/fusion_prithvi_only_log_mean_tests_training_curve.png',
    'outputs/figures/fusion_prithvi_only_log_mean_tests_regression.png',
    'outputs/figures/fusion_prithvi_only_log_mean_tests_residuals.png',
    'outputs/figures/fusion_prithvi_only_log_mean_tests_spatial.png',
    'outputs/figures/fusion_prithvi_only_log_mean_tests_binary.png',
    'outputs/figures/fusion_prithvi_only_log_mean_tests_feature_importance.png',
    # Figures — Fusion diagnostic plots
    'outputs/figures/fusion_eng_prithvi_coverage_fraction_training_curve.png',
    'outputs/figures/fusion_eng_prithvi_coverage_fraction_regression.png',
    'outputs/figures/fusion_eng_prithvi_coverage_fraction_residuals.png',
    'outputs/figures/fusion_eng_prithvi_coverage_fraction_spatial.png',
    'outputs/figures/fusion_eng_prithvi_coverage_fraction_binary.png',
    'outputs/figures/fusion_eng_prithvi_coverage_fraction_feature_importance.png',
    'outputs/figures/fusion_eng_prithvi_log_mean_tests_training_curve.png',
    'outputs/figures/fusion_eng_prithvi_log_mean_tests_regression.png',
    'outputs/figures/fusion_eng_prithvi_log_mean_tests_residuals.png',
    'outputs/figures/fusion_eng_prithvi_log_mean_tests_spatial.png',
    'outputs/figures/fusion_eng_prithvi_log_mean_tests_binary.png',
    'outputs/figures/fusion_eng_prithvi_log_mean_tests_feature_importance.png',
    # Figures — comparison
    'outputs/figures/fusion_ablation_coverage_fraction_comparison.png',
    'outputs/figures/fusion_ablation_log_mean_tests_comparison.png',
    'outputs/figures/fusion_ablation_overview.png',
    # Embeddings manifest
    'outputs/features/prithvi_frozen_embeds_manifest.json',
    # Notebook
    'notebooks/05_xgboost_prithvi_fusion.ipynb',
]
for f in files:
    if os.path.exists(f):
        subprocess.run(['git', 'add', '-f', f], check=True)
    else:
        print(f'Skipping (not yet generated): {f}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB05: Late fusion — engineered features + frozen Prithvi embeddings (XGBoost)'], check=True)
else:
    print('Nothing staged — all outputs may already be committed.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')